# Bài 3: Phân lớp MNIST bằng Neural Network và k-fold cross validation

**Họ tên:** Lê Nguyễn Anh Du  
**MSSV:** 6451071011

Thực hiện các yêu cầu a–l. Sử dụng 10 fold trên tập train gốc; chỉ dùng tập test sau khi đã chọn mô hình có accuracy validation cao nhất.

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

HO_TEN = "Lê Nguyễn Anh Du"
MSSV = "6451071011"
def in_thong_tin():
    print(f"Họ tên: {HO_TEN} | MSSV: {MSSV}")

tf.keras.utils.set_random_seed(42)
in_thong_tin()
print("Đã nhập các thư viện cần thiết và thiết lập seed.")

## a) Đọc tập train và test của MNIST từ Keras

In [ ]:
in_thong_tin()
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print("Đã tải dữ liệu MNIST từ Keras.")

## b) In kích thước tập train và test

In [ ]:
in_thong_tin()
print("train_images:", train_images.shape)
print("train_labels:", train_labels.shape)
print("test_images:", test_images.shape)
print("test_labels:", test_labels.shape)

## c) Chuẩn hóa giá trị pixel về đoạn [0, 1]

In [ ]:
in_thong_tin()
train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0
print("Khoảng pixel của tập train:", train_images.min(), "đến", train_images.max())
print("Khoảng pixel của tập test:", test_images.min(), "đến", test_images.max())

## d) Định nghĩa hàm `create_model`

Kiến trúc giống bài 2: Flatten → Dense 128 neuron, ReLU → Dropout 20% → Dense 10 neuron, Softmax. Biên dịch bằng Adam, sparse categorical crossentropy và accuracy.

In [ ]:
def create_model():
    model = Sequential([
        Input(shape=(28, 28)),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.2),
        Dense(10, activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

in_thong_tin()
print("Đã định nghĩa hàm create_model và cấu hình biên dịch.")

## e) Huấn luyện bằng 10-fold cross validation

Mỗi fold tạo một mô hình mới, huấn luyện 5 epoch với batch size 32, đánh giá trên phần validation và lưu mô hình cùng loss, accuracy của fold.

In [ ]:
in_thong_tin()
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
models = []
accuracy_per_fold = []
loss_per_fold = []
history_per_fold = []

for fold_number, (train_index, val_index) in enumerate(kfold.split(train_images), start=1):
    in_thong_tin()
    print(f"===== Fold {fold_number}/10 =====")
    X_train_fold = train_images[train_index]
    X_val_fold = train_images[val_index]
    y_train_fold = train_labels[train_index]
    y_val_fold = train_labels[val_index]

    model = create_model()
    history = model.fit(
        X_train_fold, y_train_fold,
        validation_data=(X_val_fold, y_val_fold),
        epochs=5,
        batch_size=32,
        verbose=1
    )
    fold_loss, fold_accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    models.append(model)
    history_per_fold.append(history.history)
    loss_per_fold.append(float(fold_loss))
    accuracy_per_fold.append(float(fold_accuracy))
    in_thong_tin()
    print(f"Fold {fold_number}: loss = {fold_loss:.4f}, accuracy = {fold_accuracy:.4f}")

in_thong_tin()
print(f"Đã hoàn tất {len(models)} fold và lưu {len(models)} mô hình.")

## f) Tính trung bình và độ lệch chuẩn của accuracy, loss

In [ ]:
in_thong_tin()
print(f"Mean Accuracy: {np.mean(accuracy_per_fold):.4f}")
print(f"Standard Deviation Accuracy: {np.std(accuracy_per_fold):.4f}")
print(f"Mean Loss: {np.mean(loss_per_fold):.4f}")
print(f"Standard Deviation Loss: {np.std(loss_per_fold):.4f}")

## g) Vẽ accuracy và loss theo từng fold

In [ ]:
in_thong_tin()
fold_numbers = np.arange(1, len(accuracy_per_fold) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fold_numbers, accuracy_per_fold, marker="o")
axes[0].set_title("Accuracy trên validation theo fold")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Accuracy")
axes[0].set_xticks(fold_numbers)
axes[0].grid(True)
axes[1].plot(fold_numbers, loss_per_fold, marker="o", color="tab:red")
axes[1].set_title("Loss trên validation theo fold")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("Loss")
axes[1].set_xticks(fold_numbers)
axes[1].grid(True)
plt.tight_layout()
plt.show()

## h) Chọn mô hình có accuracy validation cao nhất và dự đoán tập test

Chỉ số trong mảng bắt đầu từ 0, vì vậy số fold hiển thị bằng chỉ số cộng 1.

In [ ]:
in_thong_tin()
best_index = int(np.argmax(accuracy_per_fold))
best_model = models[best_index]
print(f"Fold tốt nhất: {best_index + 1}")
print(f"Accuracy validation của fold tốt nhất: {accuracy_per_fold[best_index]:.4f}")
test_probabilities = best_model.predict(test_images, batch_size=256, verbose=1)
test_predictions = np.argmax(test_probabilities, axis=1)
print("Số ảnh test đã phân lớp:", len(test_predictions))

## i) Accuracy của mô hình trên tập test

In [ ]:
in_thong_tin()
test_accuracy = accuracy_score(test_labels, test_predictions)
print(f"Accuracy trên tập test: {test_accuracy:.4f}")

## j) Precision, Recall, F1-score theo lớp và trung bình

In [ ]:
in_thong_tin()
print(classification_report(
    test_labels,
    test_predictions,
    labels=np.arange(10),
    target_names=[str(digit) for digit in range(10)],
    digits=4,
    zero_division=0
))

## k) Hiển thị confusion matrix bằng heatmap

In [ ]:
in_thong_tin()
cm = confusion_matrix(test_labels, test_predictions, labels=np.arange(10))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=np.arange(10), yticklabels=np.arange(10))
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.title("Confusion matrix của mô hình tốt nhất trên tập test")
plt.tight_layout()
plt.show()

## l) Phân lớp 5 ảnh đầu tiên trong tập test

In [ ]:
in_thong_tin()
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for image_index in range(5):
    image = np.expand_dims(test_images[image_index], axis=0)
    probabilities = best_model.predict(image, verbose=0)
    predicted_label = int(np.argmax(probabilities, axis=1)[0])
    true_label = int(test_labels[image_index])
    print(f"Ảnh {image_index}: dự đoán = {predicted_label}, nhãn thực tế = {true_label}")
    axes[image_index].imshow(test_images[image_index], cmap="gray")
    axes[image_index].set_title(f"Dự đoán: {predicted_label}\nThực tế: {true_label}")
    axes[image_index].axis("off")
plt.tight_layout()
plt.show()